# Build Fact Order Reviews
1. read the data from the silver order_reviews table
2. join the order_reviews table with silver orders, dim_customers and dim_date table
3. select the required columns
- Measures: review_score
- Foreign Keys:customer_sk, review_creation_date_key, review_answer_date_key
- Degenerate Dimensions: review_id, order_id 
4. write the transformed data to gold fact_order_reviews table

In [0]:
#Imports
from pyspark.sql.functions import col,cast,date_diff

### Step1 - read the data from the silver order_reviews table

In [0]:
order_reviews_df = spark.read.table("olist_catalog.silver.order_reviews")
order_reviews_df = order_reviews_df.select(
    col("review_id"),
    col("order_id"),
    col("review_score"),
    col("review_creation_date").cast("date"),
    col("review_answer_timestamp").cast("date").alias("review_answer_date")
)
display(order_reviews_df)

### Step2 - join the order_reviews table with silver orders, dim_customers and dim_date table

In [0]:
orders_df = spark.read.table("olist_catalog.silver.orders")
orders_df = (
    orders_df.select(
        col("order_id"),
        col("customer_id"),
        col("order_status"),
        col("order_purchase_timestamp").cast("date").alias("order_purchase_date"),
        col("order_approved_at").cast("date").alias("order_approved_date"),
        col("order_delivered_carrier_date").cast("date"),
        col("order_delivered_customer_date").cast("date"),
        col("order_estimated_delivery_date").cast("date"),
        date_diff(col("order_delivered_customer_date"),col("order_purchase_date")).alias("delivery_days"),
        date_diff(col("order_estimated_delivery_date"),col("order_purchase_date")).alias("estimated_delivery_days")
    )
)


In [0]:
dim_date_df = spark.read.table("olist_catalog.gold.dim_date")
dim_customers_df = spark.read.table("olist_catalog.gold.dim_customers")

In [0]:
fact_order_reviews_df = (
    order_reviews_df.alias("or")
    .join(orders_df.alias("o"),
          col("or.order_id")==col("o.order_id"),
          "left"
        )
    .join(dim_customers_df.alias("c"),
          col("o.customer_id")==col("c.customer_id"),
          "left"
        )
    .join(dim_date_df.alias("rcd"),
          col("or.review_creation_date")==col("rcd.full_date"),
          "left"
        )
    .join(dim_date_df.alias("rad"),
          col("or.review_answer_date")==col("rad.full_date"),
          "left"
        )
)

### Step3: select the required columns
- Measures: review_score
- Foreign Keys:customer_sk, review_creation_date_key, review_answer_date_key
- Degenerate Dimensions: review_id, order_id 

In [0]:
fact_order_reviews_df = (
    fact_order_reviews_df.select(
        "or.review_id",
        "or.order_id",
        "c.customer_sk",
        col("rcd.date_key").alias("review_creation_date_key"),
        col("rad.date_key").alias("review_answer_date_key"),
        "or.review_score"
    )
)

### Step4 - write the transformed data to gold fact_order_reviews table

In [0]:
(
    fact_order_reviews_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("olist_catalog.gold.fact_order_reviews")
)